# Semana 2 · Preparación y referencias

Este notebook es independiente del de la semana 1: carga y prepara de nuevo `df` y `df_test`, porque cada notebook tiene su propio kernel. Mantiene los nombres de variables que ya usábamos en `probes.ipynb`.

## Variables admitidas y excluidas

La partición temporal decide qué meses van a entrenamiento, validación y prueba. Esta tabla responde a otra pregunta: qué información conoceríamos antes de predecir el mes.

| Variable | Decisión | Motivo / momento en que se conoce |
|---|---|---|
| `month`, `channel`, `product_line`, `region`, `days_in_month` | Admitir | Calendario y características de la serie conocidos antes del mes. |
| `stock_units`, `marketing_budget`, `planned_discount`, `list_price`, `campaign` | Admitir | El diccionario los define como datos disponibles o planificados antes del mes. Los nulos en presupuesto y descuento planificado requieren una decisión justificada; nulo no significa automáticamente cero. |
| Ventas anteriores de la misma serie | Admitir cuando ya estén observadas | Son datos de meses pasados; se pueden usar retardos si están disponibles. |
| `sales_units` del propio mes | Objetivo, no predictor | Es el valor que queremos predecir. |
| `revenue` del mismo mes | Excluir | Se conoce al cierre del mes; no está disponible al hacer la predicción. |
| `website_sessions` del mismo mes | Excluir | El total se conoce al cierre del mes, después de acumular las sesiones. |
| `marketing_spend`, `avg_discount`, `closing_stock_units`, `stockout` del mismo mes | Excluir | Son resultados observados durante o al finalizar el mes. |

En particular, no usamos `revenue` ni `website_sessions` del mismo mes: si predijéramos enero, sus valores de enero todavía no estarían disponibles al empezar ese mes. Usarlos daría al análisis información posterior y falsearía el error.

In [1]:
import pandas as pd

df = pd.read_csv('../data/sales_history.csv')
df_test = df.copy()

# Se repite la preparación porque este notebook tiene un kernel separado.
# La imputación general con 0 queda provisional hasta justificar cada campo.
df_test.fillna(0, inplace=True)

,month,channel,product_line,region,marketing_spend,avg_discount,stock_units,website_sessions,sales_units,revenue,marketing_budget,planned_discount,list_price,campaign,days_in_month,closing_stock_units,stockout
0,2021-01,store,urban,coast,2190.14,0.047,150,3763.0,95,38024.70,2002.85,0.034,420.00,none,31,55,0
1,2021-01,store,urban,inland,3281.92,0.074,88,4664.0,82,31891.44,3016.71,0.062,420.00,none,31,6,0
2,2021-01,store,urban,metro,3347.16,0.050,141,5073.0,141,56259.00,3386.79,0.046,420.00,none,31,0,1
3,2021-01,store,mountain,coast,2530.35,0.037,66,3705.0,56,37210.32,2396.99,0.028,690.00,none,31,10,0
4,2021-01,store,mountain,inland,2136.16,0.101,61,3318.0,44,27293.64,2272.67,0.092,690.00,none,31,17,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1183,2026-06,ecommerce,mountain,inland,2077.34,0.042,119,5791.0,96,69895.68,2076.01,0.054,760.00,none,30,23,0
1184,2026-06,ecommerce,mountain,metro,3565.79,0.047,198,8053.0,198,143407.44,3980.64,0.039,760.00,none,30,0,1
1185,2026-06,ecommerce,electric,coast,2914.87,0.084,141,6611.0,92,137376.00,2881.99,0.073,1630.15,none,30,49,0
1186,2026-06,ecommerce,electric,inland,3349.15,0.049,82,6699.0,53,82164.45,3753.35,0.054,1630.15,none,30,29,0


In [2]:
df_ordenat = df_test.sort_values(by=['channel','product_line', 'region', 'month'], ascending=True)
conteo = df_ordenat.groupby(['channel','product_line', 'region'])['month'].size().reset_index(name='counts')
print(conteo)

      channel product_line  region  counts
0   ecommerce     electric   coast      66
1   ecommerce     electric  inland      66
2   ecommerce     electric   metro      66
3   ecommerce     mountain   coast      66
4   ecommerce     mountain  inland      66
5   ecommerce     mountain   metro      66
6   ecommerce        urban   coast      66
7   ecommerce        urban  inland      66
8   ecommerce        urban   metro      66
9       store     electric   coast      66
10      store     electric  inland      66
11      store     electric   metro      66
12      store     mountain   coast      66
13      store     mountain  inland      66
14      store     mountain   metro      66
15      store        urban   coast      66
16      store        urban  inland      66
17      store        urban   metro      66


In [3]:
df_ordenat = df_ordenat.sort_values(["channel", "product_line", "region", "month"]).copy()

grup = df_ordenat.groupby(["channel", "product_line", "region"])["sales_units"]

train = df_ordenat[df_ordenat["month"].between("2022-01", "2024-12")].copy()
val = df_ordenat[df_ordenat["month"].between("2025-01", "2025-12")].copy()
test = df_ordenat[df_ordenat["month"].between("2026-01", "2026-06")].copy()

for nombre, datos in [("Entrenamiento", train), ("Validación", val), ("Prueba", test)]:
    print(nombre, len(datos), datos["month"].min(), datos["month"].max())

Entrenamiento 648 2022-01 2024-12
Validación 216 2025-01 2025-12
Prueba 108 2026-01 2026-06


In [4]:
grupo_media = (train.groupby(["channel", "product_line", "region"])["sales_units"].mean().rename("pred_media").reset_index())

In [5]:
claves = ["channel", "product_line", "region"]

# Añadimos a cada mes las ventas de esa serie 12 meses antes.
df_con_lag = df_ordenat.copy()
df_con_lag["pred_año_anterior"] = (df_con_lag.groupby(claves)["sales_units"].shift(12))

# Nos quedamos con los meses de validación: 2025.
val_comparar = df_con_lag[df_con_lag["month"].between("2025-01", "2025-12")].copy()

# A cada fila le añadimos la media de entrenamiento de su grupo.
comparacion = val_comparar.merge(grupo_media,on=claves,how="left")

comparacion[["month", *claves, "sales_units", "pred_media", "pred_año_anterior"]].head(12)

,month,channel,product_line,region,sales_units,pred_media,pred_año_anterior
0,2025-01,ecommerce,electric,coast,60,70.222222,55.0
1,2025-02,ecommerce,electric,coast,76,70.222222,75.0
2,2025-03,ecommerce,electric,coast,57,70.222222,82.0
3,2025-04,ecommerce,electric,coast,121,70.222222,118.0
4,2025-05,ecommerce,electric,coast,83,70.222222,90.0
5,2025-06,ecommerce,electric,coast,109,70.222222,68.0
6,2025-07,ecommerce,electric,coast,102,70.222222,92.0
7,2025-08,ecommerce,electric,coast,104,70.222222,72.0
8,2025-09,ecommerce,electric,coast,81,70.222222,79.0
9,2025-10,ecommerce,electric,coast,76,70.222222,59.0


In [6]:
from sklearn.metrics import mean_absolute_error

mae_media = mean_absolute_error(
    comparacion["sales_units"],
    comparacion["pred_media"]
)

mae_anio_anterior = mean_absolute_error(
    comparacion["sales_units"],
    comparacion["pred_año_anterior"]
)

print("MAE de la media histórica:", mae_media, "unidades")
print("MAE del mismo mes del año anterior:", mae_anio_anterior, "unidades")

MAE de la media histórica: 19.92361111111111 unidades
MAE del mismo mes del año anterior: 18.944444444444443 unidades


In [7]:
# WAPE: error absoluto total como porcentaje de las ventas reales totales.
wape_media = (
    (comparacion["sales_units"] - comparacion["pred_media"]).abs().sum()
    / comparacion["sales_units"].abs().sum()
    * 100
)

wape_anio_anterior = (
    (comparacion["sales_units"] - comparacion["pred_año_anterior"]).abs().sum()
    / comparacion["sales_units"].abs().sum()
    * 100
)

print("WAPE de la media histórica:", wape_media, "%")
print("WAPE del mismo mes del año anterior:", wape_anio_anterior, "%")

WAPE de la media histórica: 19.194915254237287 %
WAPE del mismo mes del año anterior: 18.25156110615522 %


### Lectura de los errores para el negocio

El MAE indica el desvío medio en unidades por fila evaluada: por ejemplo, un MAE cercano a 19 significa que, en promedio, cada predicción de una combinación mes-canal-producto-región se aparta unas 19 unidades de la venta real. El WAPE expresa el error absoluto total como proporción de las unidades vendidas en conjunto. La referencia del mismo mes del año anterior es algo mejor en esta validación, pero el error todavía puede afectar a las decisiones de inventario; estas métricas no indican por sí solas cuánto stock comprar. La prueba de 2026 sigue reservada.